# 3.1 Document loading and chunking

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Load text, PDF, and Markdown files into LangChain `Document` objects and explain what `page_content` and `metadata` each carry.
- Split documents with `RecursiveCharacterTextSplitter` and choose `chunk_size` and `chunk_overlap` for a given corpus instead of accepting the defaults.
- Use structure-aware splitters (Markdown headers, HTML headers, JSON, code by language) so chunks respect real document boundaries.
- Describe what semantic chunking does differently, when it is worth the cost, and where it lives in the package layout.
- Evaluate a chunking strategy with chunk statistics, probe questions, and visual inspection, and recognise over-chunking and under-chunking from their symptoms.

## Installs for this lesson

In [ ]:
!pip install langchain==1.4.6 langchain-community==0.4.2 \
!  langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 \
!  pypdf

`pypdf` is only needed for PDF loading. Everything else in this lesson runs offline with no API key.

## The Document object

Every loader in LangChain returns the same shape: a list of `Document` objects from `langchain_core.documents`. A `Document` has exactly two fields:

- `page_content`: the text itself. This is what gets embedded and searched.
- `metadata`: a plain dict of everything else (source filename, page number, section heading). This is NOT embedded by default, but it travels with the chunk and is what you show the user as a citation later.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Refunds are issued within 30 days of purchase.",
    metadata={"source": "returns-policy.pdf", "page": 3},
)
print(doc.page_content)   # the text
print(doc.metadata)       # {'source': 'returns-policy.pdf', 'page': 3}

The split between the two fields matters more than it looks. Retrieval scores only see `page_content`. If you bury a crucial fact (like "this policy applies to EU customers only") in metadata, retrieval cannot match on it. If you stuff metadata-like headers into `page_content` with no structure, you get noisy chunks. A healthy habit: page_content holds coherent prose that stands on its own, metadata holds facts for filtering and citation.

## Loading: getting documents in

`TextLoader` is the simplest loader and the one to reach for first. `.load()` returns a list, even for a single file.

In [ ]:
from langchain_community.document_loaders import TextLoader

docs = TextLoader("data/refund-policy.txt").load()
print(len(docs))                  # 1: one Document per file
print(docs[0].metadata)           # {'source': 'data/refund-policy.txt'}

For PDFs, `PyPDFLoader` returns one `Document` per page, which is convenient because page numbers land in metadata automatically:

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("data/manual.pdf").load()
print(docs[0].metadata)           # {'source': 'data/manual.pdf', 'page': 0}

Markdown files can go through `TextLoader` too, but if you want the loader to understand Markdown structure, `UnstructuredMarkdownLoader` (needs the `unstructured` package) parses headings and tables more carefully. For messy scanned PDFs or Word documents where layout matters, look at `DoclingLoader` (needs the `docling` package); it is heavier but preserves tables and reading order far better than raw PDF text extraction.

To load a whole folder at once, use `DirectoryLoader`:

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

docs = DirectoryLoader(
    "data/",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()
print(len(docs), "documents loaded")

Two practical notes. First, `encoding="utf-8"` saves you from the classic Windows-1252 crash on European documents with accented characters (German umlauts, French accents); always pass it when loading text from unknown sources. Second, loaders are dumb about content quality: they will happily load duplicated pages, boilerplate footers, and navigation menus. Garbage in, garbage retrieved. A quick dedup and boilerplate pass before chunking is worth more than any clever splitter setting.

## Why chunk at all

Three forces push you to split documents, and all three are real limits, not preferences:

1. **Embedding models have a context window.** OpenAI's embedding models accept 8191 tokens per input; anything longer is silently truncated, and you pay for the whole input while only the first part is embedded. Long documents produce embeddings that represent their beginnings, not their contents.
2. **Retrieval wants small, self-contained units.** A 50-page manual embedded as one vector matches almost any query about the product, which is useless for ranking. Retrieval works best when each chunk is one idea, one answer, one paragraph-sized unit that can stand alone as evidence.
3. **The generation model has a context window.** Retrieved chunks get stuffed into the prompt. Ten 2000-character chunks cost far more tokens (and money) than five 500-character chunks, and long context makes models worse at finding the answer, not better.

So chunking is the unit-of-evidence decision for your whole RAG system. Get it wrong and retrieval returns fragments that the model cannot use, or blobs that bury the answer in noise. The social media chatter on this is unanimous on one point: most RAG failures trace back to chunking, not to the model or the vector store.

## The workhorse: RecursiveCharacterTextSplitter

This is the splitter you will use 90% of the time, and it deserves a real understanding, not a copy-pasted config. It splits text by trying separators in order, recursing into finer splits only where a piece is still too big:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
chunks = splitter.split_documents(docs)

The default separator list is `["\n\n", "\n", " ", ""]`: paragraphs first, then lines, then words, then characters as a last resort. It keeps paragraphs together when they fit, and only breaks inside a paragraph when it must. The two knobs:

- **`chunk_size`**: the target size in characters (not tokens). `len` is the default `length_function`; you can pass a token counter instead, but characters are fine for most corpora. Common starting point: 800 to 1200 characters (roughly 200 to 300 tokens). Smaller chunks mean more precise retrieval but more embedding calls and more index entries; larger chunks mean richer context per hit but noisier matching and higher prompt cost.
- **`chunk_overlap`**: how many characters the end of one chunk repeats at the start of the next. The point is to avoid cutting a sentence or an idea exactly at a boundary: the tail of chunk N becomes the head of chunk N+1, so a thought split across the boundary survives intact in at least one chunk. 10 to 20% of chunk_size is the usual range. Zero overlap is faster and cheaper but produces orphan sentences; huge overlap multiplies your embedding bill for little gain.

`add_start_index=True` adds `start_index` to each chunk's metadata: the character offset where the chunk begins in the original document. It costs nothing and is the cheapest way to reconstruct "where did this come from" later, so leave it on.

One subtlety: `chunk_size` is a target, not a guarantee. If a single paragraph is 3000 characters and your chunk_size is 1000, the splitter recurses down to words and characters and still produces chunks near the target; but it cannot split what has no separators, so pathological input (a 5000-character URL, a base64 blob) produces oversized chunks. Check your chunk statistics after splitting (below) rather than assuming.

## Splitters that respect structure

When your documents have real structure, a generic character splitter throws it away. Use the splitter that matches the format.

**Markdown.** `MarkdownHeaderTextSplitter` splits on headings and records the heading path in metadata, so each chunk knows which section it came from:

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")],
)
chunks = splitter.split_text(docs[0].page_content)
print(chunks[0].metadata)   # {'Header 1': 'Refunds', 'Header 2': 'How to request'}

This metadata is genuinely useful: you can prepend the heading path to the chunk text before embedding ("Refunds > How to request: Email support"), which fixes the common failure where a chunk says "email support" with no indication of what it is about. Watch the metadata key style though: the keys are literally `"Header 1"` and `"Header 2"` (whatever labels you pass), which is awkward in code that filters on metadata. Many teams rename them to `h1`/`h2` after splitting.

**HTML.** `HTMLHeaderTextSplitter` does the same for HTML with `headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2")]`. Note it splits on tags, it does not strip boilerplate: run your HTML through a cleaner first if it is full of nav and footer markup.

**JSON.** `RecursiveJsonSplitter` splits nested JSON while keeping the path context, so a chunk from deep inside a product catalogue still carries its category path. Useful for product data and API docs stored as JSON.

**Code.** For codebases, split on language syntax boundaries so chunks do not cut functions in half:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(code_docs)

`Language` covers 28 languages (Python, JS, TS, Java, Go, Rust, and more). The separators are language-aware: class and function definitions first, then blocks, then lines. For a code search assistant this beats character splitting by a wide margin, because a chunk that starts mid-function is nearly useless as evidence.

## Semantic chunking: splitting by meaning, not characters

All the splitters above split on structure. `SemanticChunker` splits on meaning: it embeds each sentence, measures how similar neighbouring sentences are, and cuts where the topic shifts. It lives in the experimental package:

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",  # or "standard_deviation", "interquartile", "gradient"
)
chunks = splitter.split_documents(docs)

The `breakpoint_threshold_type` decides how aggressive the cuts are: `"percentile"` (the default) cuts at the largest similarity drops, producing fewer, larger chunks; `"gradient"` is more aggressive and produces many small chunks. `buffer_size` (default 1) controls how many neighbouring sentences are merged before embedding, which smooths the similarity signal.

Be honest about the cost before using it. Semantic chunking embeds every sentence of your corpus, so it costs an embedding pass over the full text just to decide where to cut, and it is slow on large document sets. Use it when your corpus mixes topics within pages (long reports, transcripts, research papers) and fixed-size chunking keeps slicing mid-argument. For well-structured docs (manuals with headings, Markdown files), a header-aware splitter is cheaper and usually better.

## Metadata: the part people skip

Splitters propagate loader metadata to every chunk (source file, page number), and `add_start_index` adds the character offset. Add your own metadata deliberately, because it is the foundation of everything downstream:

- **Source and location** (file, page, URL, section path): for citations in the final answer and for debugging retrieval ("why did this chunk match?").
- **Timestamps and versions** (document date, version): for filtering stale content later ("only policies updated after 2025").
- **Access level** (public, internal, confidential): if your agent serves multiple users, you will need metadata filtering so retrieval never returns documents the asker is not allowed to see. Design this in now; bolting permissions onto a RAG system later is painful.

Keep metadata values flat (strings, numbers, booleans). Nested dicts and lists work in some stores and break in others; flat metadata works everywhere.

## Evaluating chunking: prove it works

Chunking is the one part of RAG where "it looks fine" is not good enough, because bad chunking fails silently: retrieval just returns slightly worse chunks and the model writes slightly worse answers. Three checks, in order of effort:

**1. Chunk statistics.** After splitting, print the distribution. You are looking for surprises, not averages:

In [ ]:
lengths = [len(c.page_content) for c in chunks]
print(f"{len(chunks)} chunks | min {min(lengths)} | max {max(lengths)} | "
      f"mean {sum(lengths)/len(lengths):.0f}")

# Flag the suspicious ones: tiny fragments and oversized blobs
tiny = [c for c in chunks if len(c.page_content) < 100]
huge = [c for c in chunks if len(c.page_content) > 1500]
print(f"{len(tiny)} tiny chunks, {len(huge)} oversized chunks")
for c in (tiny + huge)[:5]:
    print(repr(c.page_content[:120]), "|", c.metadata.get("source"))

Tiny chunks are usually headers, list items, or table fragments that split badly; oversized chunks are the base64-blob problem. Both poison retrieval: tiny chunks match queries on almost nothing and contribute nothing as evidence, oversized chunks match everything and cost tokens.

**2. Probe questions.** Write 10 to 20 questions you know the answers to, each answerable by one specific passage in your corpus. Embed your chunks, retrieve top-3 for each question, and check whether the right passage is in the results. This "hit rate" is the single most honest metric for chunking quality. If a probe fails, look at what was retrieved instead: the failure is usually a chunk boundary that split the answer from its context, which tells you exactly what to fix (more overlap, a structure-aware splitter, or heading context prepended to the chunk).

**3. Visual inspection.** Print 10 random chunks with their metadata and read them as if you were the model receiving them as evidence. Ask: could I answer a question from this chunk alone? Does it say what document and section it is from? The random sample catches the embarrassing cases (a chunk that is just "click here", a chunk that starts mid-sentence with no subject) that statistics miss.

## Over-chunking and under-chunking: the two failure modes

- **Over-chunking** (chunks too small): retrieval returns fragments. The model gets five chunks that each contain half a thought and cannot assemble the answer. Symptom in probe tests: the right information is "nearby" but never in the top-k as a complete unit. Common cause: chunk_size below 300 characters on prose, or aggressive semantic splitting.
- **Under-chunking** (chunks too large): retrieval returns blobs. The answer is technically in the retrieved chunk, but buried in 2000 characters of unrelated text, so the model either misses it or quotes irrelevant context. Symptom: high probe hit rate but poor answer quality and high token bills. Common cause: chunk_size above 2000 characters, or loaders that return whole PDFs as single documents.

The fix is rarely just the size knob. Over-chunking on manuals usually means you need header-aware splitting (each section becomes a chunk, whatever its size). Under-chunking on transcripts usually means semantic chunking. Start with the recursive splitter at 1000/200, run the probe set, and let the failures tell you which direction to move.

## What can go wrong

- **Silent truncation by the embedding model.** Any chunk over 8191 tokens gets truncated before embedding, and OpenAI charges for the full input. If your chunks are oversized, you are paying for text that never reaches the vector. Keep chunk_size well under the limit and watch the `huge` list from the statistics check.
- **Metadata lost in transit.** Some stores and some serialisation paths drop metadata keys with spaces or non-string values. Keep metadata flat and simple from the start, and verify after a round trip through your vector store (load a document back and compare metadata).
- **Re-chunking means re-embedding.** Every chunking change invalidates every embedding you have already paid for. Chunking is not free to iterate on: settle the strategy on a sample of your corpus with probe questions before embedding the full set.
- **Encoding crashes on European text.** Files from Windows systems often use Windows-1252 or Latin-1, and `TextLoader` without an explicit encoding raises on the first umlaut or accent. Pass `encoding="utf-8"` (or detect with a library) in `loader_kwargs` via `DirectoryLoader`.
- **PDF text extraction is lossy.** `PyPDFLoader` extracts raw text: tables become jumbled columns, two-column layouts interleave, scanned PDFs return nothing at all. If your corpus is PDF-heavy, budget time for extraction quality (try `DoclingLoader` on the worst files) before blaming the splitter.
- **Duplicate content inflates the index.** Repeated boilerplate (headers, footers, disclaimers on every page) becomes dozens of near-identical chunks that crowd out real results. Deduplicate after splitting: identical or near-identical `page_content` with different page numbers is almost always boilerplate.
- **SemanticChunker cost and speed.** It embeds every sentence to decide boundaries, which doubles your embedding spend on the corpus and is slow on large sets. Reserve it for unstructured prose where fixed-size splitting demonstrably fails.

## Check your understanding

**1. A `Document` has `page_content` and `metadata`. Why does it matter that the two are separate, and what goes wrong if you mix them up?**
Only `page_content` is embedded and matched during retrieval; `metadata` travels along for filtering and citation but is invisible to similarity search. If you put a crucial fact only in metadata (like which product a chunk describes), no query can match on it. If you stuff headers, URLs, and boilerplate into `page_content`, the embedding represents noise and retrieval quality drops.

**2. Explain what `chunk_overlap` is for, and why setting it to zero is usually a mistake.**
Overlap repeats the tail of one chunk at the head of the next, so an idea cut at a boundary survives intact in at least one chunk. With zero overlap, sentences and arguments sliced at boundaries become orphan fragments: the end of chunk N and the start of chunk N+1 each lack the context the other half holds, and retrieval can return a chunk that is unusable as evidence.

**3. Your corpus is Markdown documentation with clear headings, and probe questions keep failing because retrieved chunks say things like "email support" with no indication of what they refer to. What do you change?**
Switch to `MarkdownHeaderTextSplitter` with `headers_to_split_on` matching the heading levels, so each chunk carries its heading path in metadata. Then prepend that path to the chunk text before embedding (for example, "Refunds > How to request: email support"), which makes every chunk self-describing. Optionally rename the metadata keys to something code-friendly like `h1`/`h2`.

**4. When is `SemanticChunker` worth its cost, and where does it live in the package layout?**
It is worth it when the corpus mixes topics within pages (transcripts, long reports, research papers) and fixed-size or structure-aware splitting keeps cutting mid-argument; for well-structured docs, header-aware splitting is cheaper and better. It lives in `langchain_experimental.text_splitter` (experimental, needs `langchain-experimental` installed), and it costs a full embedding pass over every sentence just to decide boundaries, so it roughly doubles embedding spend and is slow on large sets.

**5. After splitting you find 40 chunks under 100 characters and 12 chunks over 3000 characters. What are the likely causes, and what do you do about each?**
The tiny chunks are usually headers, list items, or table fragments that split badly; filter or merge them (merge short chunks with neighbours, or drop pure boilerplate). The oversized chunks are input the splitter could not break (long URLs, base64 blobs, unbroken text); inspect them, and if they are junk, exclude them before embedding, since they also risk silent truncation by the embedding model. Re-run the statistics check after fixing, and do this on a sample before embedding the full corpus, because re-chunking invalidates paid embeddings.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.